# Financial Behaviour & Insights Engine

Builds the first Insights Engine for Finance Analytics
(`finance_analytics.insights`): combines PR-009's anomaly results, PR-010's
recurring-transaction results and PR-011's merchant/category enrichment,
plus two new families of period-comparison rules of its own, into a small
set of structured, deterministic `Insight` objects.

```
Transactions
     |
Deduplication
     |
     +-- Enrichment (PR-011)          --> category_change_insights
     +-- Anomaly detection (PR-009)   --> unusual_transaction_insights
     +-- Recurring detection (PR-010) --> recurring_payment_insights
     +-- spending_trend / income_expense_change / savings_rate_change (new)
     |
     v
Deduplication -> Ranking -> Structured Insights
```

**Critical principle this notebook follows throughout (PR-012):** every
insight must trace back to *Evidence -> Analytical rule -> Insight*. No
LLM, no invented facts, no financial advice, no fraud claims, no
unsupported predictions. Anomaly detection and recurring-transaction
classification are **consumed**, not reimplemented — see
`notebooks/03_anomaly_detection.ipynb` and
`notebooks/04_recurring_transactions.ipynb` for how those results were
produced and validated.


In [1]:
import json
import random
from dataclasses import replace
from pathlib import Path

import pandas as pd

from finance_analytics.anomalies.detector import detect_anomalies
from finance_analytics.insights.deduplication import deduplicate_insights
from finance_analytics.insights.engine import generate_insights
from finance_analytics.insights.periods import build_period_comparison
from finance_analytics.insights.ranking import rank_insights
from finance_analytics.insights.rules import (
    category_change_insights,
    income_expense_change_insights,
    savings_rate_change_insights,
    spending_trend_insights,
)
from finance_analytics.io.csv import load_transactions_csv
from finance_analytics.recurring.detector import detect_recurring_transactions

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

## 1. Input: A Clean Transaction Set

Loads the same synthetic fixture used throughout this project and applies
the same three-step cleaning notebooks 02-05 use before behavioural
analysis (`notebooks/02_exploratory_data_analysis.ipynb`, section 3):

1. Drop rows with a structurally invalid date or amount (rows 19, 20).
2. Drop the duplicate transaction id (row 18 duplicates row 14).
3. Drop the QA-fixture row disguised as a transaction (row 21 — a
   structurally valid row whose `description` says "Missing merchant
   test").

`finance_analytics.insights.engine.generate_insights` itself only
deduplicates by `id` (step 2) — it does not special-case "test" rows,
since that convention is specific to this fixture, not a general product
rule. Steps 1 and 3 are already handled inside PR-009/PR-010/PR-011's own
functions (invalid dates/amounts are dropped there) or are this
notebook's own choice, matching the precedent of notebooks 02-05, so the
insights shown below are computed over genuine behavioural data rather
than QA scaffolding.


In [2]:
transactions = load_transactions_csv(DATA_PATH)
clean = transactions.dropna(subset=["date", "amount"]).drop_duplicates(subset=["id"])
clean = clean[~clean["description"].fillna("").str.contains("test", case=False)]

print(f"Raw rows: {len(transactions)}  |  Clean rows: {len(clean)}")
clean

Raw rows: 22  |  Clean rows: 18


,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account
5,6,2026-02-12,-18.75,EUR,Pharmacy,Farmácia Central,Health,Main Account
6,7,2026-02-15,-120.00,EUR,Train tickets,CP,Transport,Main Account
7,8,2026-02-18,-642.50,EUR,"Flight, Lisbon to Rome",TAP Air,Travel,Main Account
8,9,2026-02-20,-899.00,EUR,New laptop purchase,MediaMarkt,Shopping,Main Account
9,10,2026-02-22,-8.20,EUR,"Lunch, ""daily menu""",Café Central,Food & Dining,Main Account


## 2. Anomaly & Recurring Signals (PR-009 / PR-010) — Consumed, Not Reimplemented

`finance_analytics.insights.conversions` adapts these results into
`Insight` objects without recomputing any score or threshold. This section
runs `detect_anomalies` / `detect_recurring_transactions` directly — the
same calls the engine makes internally — so the raw evidence is visible
before it becomes an insight.


In [3]:
anomaly_results = detect_anomalies(clean)
flagged_anomalies = [r for r in anomaly_results if r.is_anomaly]

print(
    f"{len(anomaly_results)} expense transactions scored, {len(flagged_anomalies)} flagged as anomalous"
)
pd.DataFrame(
    [
        {
            "transaction_id": r.transaction_id,
            "method": r.method,
            "anomaly_score": r.anomaly_score,
            "reason": r.reason,
        }
        for r in flagged_anomalies
    ]
)

16 expense transactions scored, 4 flagged as anomalous


,transaction_id,method,anomaly_score,reason
0,7,global_relative_robust_z,3.08,Amount is 3.2× higher than your typical transa...
1,8,global_relative_robust_z,15.35,Amount is 14.3× higher than your typical trans...
2,9,global_relative_robust_z,17.99,Amount is 18.0× higher than your typical trans...
3,15,category_relative_robust_z,3.97,Amount is 3.0× higher than your typical Food &...


In [4]:
recurring_results = detect_recurring_transactions(clean)
notable_recurring = [
    r for r in recurring_results if r.classification in {"Recurring", "Possible recurring"}
]

print(
    f"{len(recurring_results)} merchants classified, "
    f"{len(notable_recurring)} 'Recurring'/'Possible recurring'"
)
pd.DataFrame(
    [
        {
            "merchant": r.merchant,
            "classification": r.classification,
            "confidence_score": r.confidence_score,
            "occurrences": r.occurrences,
            "reason": r.reason,
        }
        for r in notable_recurring
    ]
)

13 merchants classified, 2 'Recurring'/'Possible recurring'


,merchant,classification,confidence_score,occurrences,reason
0,Continente,Possible recurring,0.5471,2,"Continente appears 2 times, about every ~28 da..."
1,Spotify,Possible recurring,0.6250,2,"Spotify appears 2 times, about every ~28 days ..."


As `notebooks/03_anomaly_detection.ipynb` and
`notebooks/04_recurring_transactions.ipynb` already documented in detail:
`MediaMarkt` and `TAP Air` are large one-off purchases flagged against the
global baseline; `CP` (Transport) is flagged against the same baseline at a
more modest z-score; `O Pescador` is flagged *category*-relative (Food &
Dining mixes a coffee, a lunch and two dinners, so one dinner reads as
unusual against that mixed baseline — a known category-granularity
limitation, not a detector bug). `Spotify` and `Continente` are this
fixture's only repeat merchants and both reach "Possible recurring" — never
"Recurring", since neither has the 3rd occurrence needed to confirm
interval consistency (PR-010's structural floor).


## 3. Merchant/Category Enrichment (PR-011) Feeds the Category-Change Rule

`generate_insights` calls `enrichment.models.enrich_transactions` once and
uses its `category` field for `category_change_insights` — this is the
"future analytics" PR-011 built its enrichment layer for. `detect_anomalies`
and `detect_recurring_transactions` are left untouched: they still consume
the original raw `category`/`merchant` columns exactly as PR-009/PR-010
validated them, so this PR does not risk changing their already-tested
behaviour. `notebooks/05_merchant_and_category_analysis.ipynb` found
enrichment to be a verified no-op on this fixture's `category` column, so
using enriched vs. raw category makes no numeric difference here — but the
integration point is real and will matter for a less clean dataset.


## 4. Why Month-Over-Month Comparison Needs Care Here

PR-008's EDA found this fixture's two observed months are not directly
comparable: February is a full month; March is not (the export ends
partway through it). PR-012 §8 requires the same discipline generally: "Do
not compare non-comparable periods... use comparable partial-period
analysis or suppress the insight."

`finance_analytics.insights.periods.build_period_comparison` resolves this
automatically — full month vs full month when nothing suggests the export
stopped early, otherwise both months truncated to the same day-of-month
cutoff. There is no wall-clock "today" in a CSV export, so the latest
transaction date is the only signal available.


In [5]:
comparison = build_period_comparison(clean)
comparison

PeriodComparison(current_period='2026-03', previous_period='2026-02', is_current_period_complete=False, day_cutoff=15, comparison_kind='comparable_partial')

**Observed:** March's last clean transaction is on day 15 (`id=18`, the
freelance payment) — not day 19 as in the raw file, because the QA-fixture
row (`id=21`, dated 2026-03-19) was excluded during cleaning (section 1).
March has 31 days, so day 15 is not the last calendar day: the comparison
is resolved as **comparable partial** — both February and March are
truncated to their first 15 days before any rule compares them. This is
the same principle the EDA applied qualitatively, now enforced
automatically and generically (it does not hard-code "March 2026").


## 5. Insight Rules (PR-012 §§2-3)

| Insight type | Rule | Threshold | Source |
|---|---|---|---|
| Spending Trend | Total expenses, current vs previous comparable window | ±15% | `rules.spending_trend_insights` |
| Category Change | Per-category expenses, current vs previous comparable window (>=2 transactions per window) | ±25% | `rules.category_change_insights` |
| Income / Expense Change | Expenses move meaningfully while income does not move the same way | ±15% (expense), ±5% (income "stable" band) | `rules.income_expense_change_insights` |
| Savings Rate Change | `(income - expenses) / income`, percentage-point change | ±10pp | `rules.savings_rate_change_insights` |
| Unusual Transaction | PR-009's flagged anomalies | *(PR-009's own `ANOMALY_Z_THRESHOLD=3.0`)* | `conversions.unusual_transaction_insights` |
| Recurring Payment | PR-010's `"Recurring"` / `"Possible recurring"` classifications | *(PR-010's own thresholds)* | `conversions.recurring_payment_insights` |

None of the four new thresholds are statistically fitted — like every
other threshold in this codebase, they are documented, evidence-motivated
choices (see `rules.py`'s module docstring), not values tuned against
labelled outcomes (none exist for this product).


In [6]:
print("Spending trend:")
for insight in spending_trend_insights(clean):
    print(f"  {insight.description}")

print("\nCategory change:")
category_results = category_change_insights(clean)
if not category_results:
    print("  (none — see explanation below)")
for insight in category_results:
    print(f"  {insight.description}")

print("\nIncome / expense change:")
for insight in income_expense_change_insights(clean) or ["  (none — see explanation below)"]:
    print(insight if isinstance(insight, str) else f"  {insight.description}")

print("\nSavings rate change:")
for insight in savings_rate_change_insights(clean) or ["  (none — see explanation below)"]:
    print(insight if isinstance(insight, str) else f"  {insight.description}")

Spending trend:
  Your spending decreased 31% compared with the same 15-day period last month.

Category change:
  (none — see explanation below)

Income / expense change:
  (none — see explanation below)

Savings rate change:
  (none — see explanation below)


**Observed, and why each empty result is honest rather than a bug:**

- **Spending trend fires.** Expenses fell from €363.54 (Feb 1-15) to
  €251.49 (Mar 1-15) — a real, evidence-based -31% change, comfortably past
  the 15% threshold.
- **Category change: none.** Every category in this fixture has exactly
  one transaction per comparable 15-day window (`Groceries`,
  `Subscriptions`, `Food & Dining`, `Transport`, ...) — below
  `MIN_CATEGORY_TRANSACTIONS=2`. This mirrors the precedent
  `notebooks/04_recurring_transactions.ipynb` already set: real data here
  is too thin for a category-level comparison to be trustworthy, so the
  rule correctly stays silent rather than reporting a swing driven by one
  purchase. Section 11 demonstrates this rule firing on a synthetic
  fixture with enough evidence.
- **Income / expense change and savings rate change: both suppressed.**
  February 1-15 (the comparable previous window) has **zero income** — the
  one salary payment lands on 2026-02-28, outside the 15-day cutoff. Both
  rules require income in the previous window to define a baseline (a
  percent change or a savings rate against zero income is undefined, not
  merely large), so both are suppressed rather than reporting a nonsensical
  number. Section 11 demonstrates both rules firing synthetically.


## 6. Full Pipeline: Generated Structured Insights

`generate_insights` runs every rule above, adapts the anomaly/recurring
results, deduplicates, and returns a single ranked list.


In [7]:
insights = generate_insights(clean)

print(f"{len(insights)} insights generated\n")
pd.DataFrame([i.to_dict() for i in insights])[["id", "type", "severity", "confidence", "title"]]

7 insights generated



,id,type,severity,confidence,title
0,unusual_transaction:8,unusual_transaction,IMPORTANT,0.6000,Unusual transaction
1,unusual_transaction:9,unusual_transaction,IMPORTANT,0.6000,Unusual transaction
2,spending_trend:2026-03,spending_trend,NOTICE,0.7600,Spending decreased
3,unusual_transaction:15,unusual_transaction,NOTICE,0.7500,Unusual transaction
4,unusual_transaction:7,unusual_transaction,NOTICE,0.6000,Unusual transaction
5,recurring_payment:Spotify:EUR,recurring_payment,INFO,0.6250,Possible recurring: Spotify
6,recurring_payment:Continente:EUR,recurring_payment,INFO,0.5471,Possible recurring: Continente


**Observed:** 7 insights survive from this 18-row fixture — 2 `IMPORTANT`
anomalies (`MediaMarkt`, `TAP Air`), 1 `NOTICE` spending trend, 2 more
`NOTICE` items (`CP`'s anomaly, `O Pescador`'s category-relative anomaly),
and 2 `INFO` recurring-payment candidates (`Spotify`, `Continente`). No
`category_change`, `income_expense_change` or `savings_rate_change`
insight appears — consistent with section 5's evidence-based suppression,
not a missing feature.


## 7. Ranking (PR-012 §6)

Insights are ranked by `(severity desc, confidence desc, recency desc, id
asc)` using sequential stable sorts (`ranking.rank_insights`) — no learned
weights, no recommendation model. `generate_insights` already returns a
ranked list; this cell verifies that re-ranking a shuffled copy reproduces
the exact same order, i.e. the ordering depends only on each insight's own
fields, never on generation order.


In [8]:
shuffled = insights.copy()
random.Random(42).shuffle(shuffled)
reranked = rank_insights(shuffled)

assert [i.id for i in reranked] == [i.id for i in insights]
print("Re-ranking a shuffled copy reproduces the same order — ranking is deterministic.")

Re-ranking a shuffled copy reproduces the same order — ranking is deterministic.


## 8. Deduplication (PR-012 §7)

This fixture's insights are already unique by construction (each rule
produces at most one insight per `(type, category, merchant,
comparison_period)` key), so there is nothing to deduplicate here. This
cell manufactures a synthetic duplicate — a weaker copy of a real insight
sharing the same key — to demonstrate the mechanism itself.

That key only applies to period-comparison insights (`spending_trend`,
`category_change`, `income_expense_change`, `savings_rate_change`) —
`deduplication.py`'s module docstring documents why conversion-based
insights (`unusual_transaction`, `recurring_payment`) are deliberately
excluded and deduplicated on their own already-unique `id` instead: two
distinct anomalous transactions (or a merchant billed in more than one
currency) are genuinely separate facts, not duplicates. So the demo needs
a period-comparison insight specifically, not just `insights[0]` — this
fixture's only one is `spending_trend` (section 6).

In [9]:
original = next(i for i in insights if i.type == "spending_trend")
weaker_duplicate = replace(
    original,
    id="synthetic_weaker_duplicate",
    confidence=round(max(0.0, original.confidence - 0.3), 2),
)

deduped = deduplicate_insights([weaker_duplicate, original])

print("Input: 2 insights sharing (type, category, merchant, comparison_period)")
print(f"Output: {len(deduped)} kept -> id={deduped[0].id!r}, confidence={deduped[0].confidence}")
assert deduped[0].id == original.id  # the higher-confidence insight survives

Input: 2 insights sharing (type, category, merchant, comparison_period)
Output: 1 kept -> id='spending_trend:2026-03', confidence=0.76


## 9. Example Explanations

Every insight's `description` is generated by a fixed template (this PR's
`rules.py`) or taken verbatim from PR-009/PR-010's own deterministic
explanations (`conversions.py`) — no LLM anywhere in this pipeline. The
same input always produces the same sentence.


In [10]:
for insight in insights:
    print(f"[{insight.severity}] {insight.title}")
    print(f"  {insight.description}")
    print(f"  confidence={insight.confidence}  comparison_period={insight.comparison_period}")
    print()

[IMPORTANT] Unusual transaction
  Amount is 14.3× higher than your typical transaction (usually around 45.00 EUR).
  confidence=0.6  comparison_period=None

[IMPORTANT] Unusual transaction
  Amount is 18.0× higher than your typical transaction (usually around 49.95 EUR).
  confidence=0.6  comparison_period=None

[NOTICE] Spending decreased
  Your spending decreased 31% compared with the same 15-day period last month.
  confidence=0.76  comparison_period=2026-03

[NOTICE] Unusual transaction
  Amount is 3.0× higher than your typical Food & Dining transaction (usually around 12.50 EUR).
  confidence=0.75  comparison_period=None

[NOTICE] Unusual transaction
  Amount is 3.2× higher than your typical transaction (usually around 37.49 EUR).
  confidence=0.6  comparison_period=None

[INFO] Possible recurring: Spotify
  Spotify appears 2 times, about every ~28 days with a similar amount around 29.99 EUR, but at least 3 occurrences are needed to confirm a stable interval.
  confidence=0.625  c

## 10. Example Output Suitable for an API (PR-012 §16)

`Insight.to_dict()` renders a plain, JSON-serialisable dict — the shape a
future FastAPI response body would return as-is. No FastAPI/HTTP layer is
implemented in this PR (see Out of Scope below); this only demonstrates
the model is already shaped for one.


In [11]:
api_payload = {"insights": [i.to_dict() for i in insights[:3]]}
print(json.dumps(api_payload, indent=2))

{
  "insights": [
    {
      "id": "unusual_transaction:8",
      "type": "unusual_transaction",
      "title": "Unusual transaction",
      "description": "Amount is 14.3\u00d7 higher than your typical transaction (usually around 45.00 EUR).",
      "severity": "IMPORTANT",
      "confidence": 0.6,
      "related_transaction_ids": [
        "8"
      ],
      "metadata": {
        "anomaly_score": 15.35,
        "method": "global_relative_robust_z",
        "reference_context": {
          "merchant_median": null,
          "merchant_mad": null,
          "merchant_transaction_count": 0,
          "category_median": null,
          "category_mad": null,
          "category_iqr": null,
          "category_transaction_count": 0,
          "global_median": 45.0,
          "global_mad": 26.25,
          "global_transaction_count": 7
        },
        "source_feature": "anomalies.detect_anomalies"
      },
      "category": "Travel",
      "merchant": "TAP Air",
      "amount": -642.5,
 

## 11. Controlled Scenarios (PR-012 §14)

The real fixture (sections 5-6) only exercises `spending_trend` and, via
PR-009/PR-010, `unusual_transaction`/`recurring_payment` — it is too small
to demonstrate `category_change`, `income_expense_change` or
`savings_rate_change` firing. `tests/test_insights_engine.py` covers all
five documented scenarios (A: stable spending -> no insight; B: significant
increase -> spending-trend insight; C: unusual transaction -> anomaly
insight; D: recurring subscription -> recurring insight; E: insufficient
history -> no insight) plus `tests/test_insights_rules.py`'s per-rule
increase/decrease/stable/insufficient-data cases end to end. This cell
demonstrates one additional synthetic scenario inline — a full pipeline run
where a category swing, an income/expense divergence and a savings-rate
change all have enough evidence to fire together, with a second, stable
category alongside it and no incidental anomaly/recurring noise (every
merchant name below is used exactly once, so PR-009/PR-010's own minimum-
history rules correctly leave them all unscored/unclassified).


In [12]:
def synthetic_row(id_, date, amount, category, merchant):
    return {
        "id": id_,
        "date": pd.Timestamp(date),
        "amount": amount,
        "currency": "EUR",
        "description": "synthetic",
        "merchant": merchant,
        "category": category,
        "account": "Main Account",
    }


synthetic_rows = [
    # January (previous, full-month baseline)
    synthetic_row("g1", "2026-01-05", -25.0, "Groceries", "Fresh Foods"),
    synthetic_row("h1", "2026-01-10", -15.0, "Health", "City Clinic"),
    synthetic_row("g2", "2026-01-15", -35.0, "Groceries", "Corner Market"),
    synthetic_row("h2", "2026-01-20", -25.0, "Health", "Pharmacy One"),
    synthetic_row("inc1", "2026-01-28", 200.0, "Income", "Employer"),
    # February (current — last transaction on day 24; February's real last
    # day is 28, so this is itself a comparable-partial comparison too)
    synthetic_row("g3", "2026-02-03", -46.0, "Groceries", "City Grocer"),
    synthetic_row("h3", "2026-02-08", -18.0, "Health", "MedCenter"),
    synthetic_row("g4", "2026-02-14", -48.0, "Groceries", "SuperValue"),
    synthetic_row("h4", "2026-02-24", -22.0, "Health", "HealthPlus"),
    synthetic_row("inc2", "2026-02-28", 202.0, "Income", "Employer"),
]
synthetic_frame = pd.DataFrame(synthetic_rows)

synthetic_insights = generate_insights(synthetic_frame)
pd.DataFrame([i.to_dict() for i in synthetic_insights])[
    ["id", "type", "severity", "confidence", "title", "description"]
]

,id,type,severity,confidence,title,description
0,income_expense_change:2026-02,income_expense_change,NOTICE,1.00,Expenses and income diverged,Your expenses increased 34% while income remai...
1,spending_trend:2026-02,spending_trend,NOTICE,1.00,Spending increased,Your spending increased 34% compared with last...
2,category_change:Groceries:2026-02,category_change,NOTICE,0.87,Groceries spending increased,Your Groceries spending increased 57% compared...
3,savings_rate_change:2026-02,savings_rate_change,INFO,1.00,Savings rate decreased,Your savings rate decreased from 50% to 34% co...


**Observed:** with enough evidence per window, `category_change` fires for
`Groceries` (€60 -> €94, +57%) but correctly stays silent for `Health`
(€40 -> €40, unchanged); `spending_trend` fires (+34% overall);
`income_expense_change` fires (expenses up 34% while income moves <1%, well
inside the "stable" band); `savings_rate_change` fires (50% -> 34%, a 16
percentage-point drop). No `unusual_transaction` or `recurring_payment`
insight appears — every merchant here is seen exactly once, so PR-009's
minimum-history gate and PR-010's minimum-occurrence gate both correctly
leave them unscored, exactly as they would for any real user's first
transaction with a new merchant. This confirms the suppressions observed in
section 5 are a property of *this project's fixture* being small and
sparse, not of the rules themselves.


## 12. Limitations

- **The real fixture supports only one comparable-partial window.** Every
  period-based conclusion above (section 6) rests on a single 15-day
  Feb-vs-Mar comparison — not enough to say anything about a real trend,
  only to demonstrate the mechanism correctly on genuine data. This mirrors
  the precedent notebooks 03/04 already set for their own detectors.
- **`category_change`, `income_expense_change` and `savings_rate_change`
  are demonstrated only synthetically on real project data** (sections 5,
  11) — the fixture is too sparse (1 transaction/category/window, no
  income before day 15 of either month) to exercise them for real. This is
  a data-size limitation, not evidence the rules are wrong, exactly the
  same conclusion `notebooks/04_recurring_transactions.ipynb` reached about
  its own "Recurring" tier.
- **Unusual-transaction and recurring-payment insights inherit every
  limitation already documented in notebooks 03 and 04** — cold-start,
  no ground truth, evidence-motivated (not statistically fitted)
  thresholds, category-granularity effects (`O Pescador`).
- **Confidence is a heuristic score, not a calibrated probability**,
  everywhere in this module — for period-comparison insights
  (`periods.comparison_confidence`), for anomaly insights (a fixed score
  per detection tier) and for recurring insights (taken directly from
  PR-010's own `confidence_score`). None of these have been fit or
  validated against labelled outcomes.
- **Ranking and deduplication are exercised on a small N.** With 7 real
  insights there are few genuine ties to observe — `tests/test_insights_ranking.py`
  and `tests/test_insights_deduplication.py` cover the tie-breaking logic
  with dedicated synthetic cases the notebook does not repeat.
- **No ground truth for "is this insight useful".** As with every prior
  analytical feature in this project, there is no labelled dataset saying
  which insights a real user would find valuable — evaluation here is
  controlled scenarios and manual inspection, not an accuracy claim.
- **Synthetic/test data.** As in every notebook in this project: this
  fixture exists to exercise code paths, not to represent real spending
  behaviour.


## Out of Scope — Confirmation

Not implemented in this notebook or in `finance_analytics.insights`: a
FastAPI service or REST endpoints, Android integration, Room persistence,
LLM-generated insights (every `description` above is either a fixed
template or PR-009/PR-010's own deterministic text), financial advice,
investment recommendations, fraud claims, unsupported predictions, or a
complex recommendation model (`ranking.py` only orders insights that
already exist). `finance_analytics.anomalies`, `finance_analytics.recurring`
and `finance_analytics.enrichment` are unmodified by this notebook — every
result shown in sections 2-3 comes from calling their existing, already-
tested functions directly.
